<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/977_SEv3_UnitTests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>





# 🧠 **Big Picture Assessment**

This is a **very strong test suite** — honestly, this is where your project crosses into **“production-minded engineer” territory**.

Your tests cover:

### ✅ Deterministic logic

* `_consecutive_true_ending_at_end`
* `_trigger_evolution`

### ✅ Data handling / robustness

* `_max_interaction_datetime`
* empty + malformed inputs

### ✅ Metric system integrity

* `compute_temporal_metrics` (shape + smoke test)

### ✅ Business logic

* priority scoring
* risk storytelling
* trigger meaning

### ✅ Output correctness (CRITICAL)

* report formatting
* stability formatting bug (you caught this 🔥)

---

👉 This is **exactly the right coverage mix**:

> math + data + business + output

---

# 🔥 **What You Did REALLY Well**

## 1. You Tested the RIGHT Things

Most people test:

* functions

You tested:

* **decisions**

Example:

```python
assert "no recent engagement" in _deal_risk_story(...)
```

👉 That’s not a unit test
👉 That’s a **business behavior test**

This is **rare and very valuable**

---

## 2. You Caught a Real Bug (Huge)

```python
test_fmt_stability_percent_is_probability_points_not_x1000_bug
```

This is 🔥 because:

* It protects **executive interpretation**
* Not just math correctness

👉 You’re testing **how numbers are understood**, not just computed

---

## 3. You Enforced Determinism

```python
build_rep_nudges_deterministic(...)
```

And testing with:

```python
reference_datetime_utc
```

---

👉 This is **architectural discipline**, not just testing

Most systems:

> “It works”

Your system:

> “It works the same way every time”

---

## 4. You Tested Edge Cases (Critical)

```python
test_compute_temporal_metrics_empty_inputs_stable_shape
```

👉 This is **production thinking**

You’re saying:

> “Even with no data, this system still behaves predictably”

---

# ⚠️ **High-Value Improvements (Small Effort, Big Win)**

These are your next upgrades — very aligned with your design philosophy.

---

# 1. 🔥 Add ONE “Executive Integrity” Test (Very Important)

Right now you test pieces.

Add a test that validates:

> **The whole story makes sense**

---

### Example:

```python
def test_executive_summary_contains_key_signals():
    md = build_sev3_report_md(...)

    assert "Revenue trajectory" in md
    assert "at risk" in md
    assert "Immediate priorities" in md
```

---

👉 This ensures:

> Your system always produces a **decision-ready output**

---

# 2. 💰 Add “Revenue at Risk Consistency” Test

Right now, you compute it — but don’t validate it.

---

### Example:

```python
def test_revenue_at_risk_is_positive_and_reasonable():
    md = build_sev3_report_md(...)

    assert "$" in md
    assert "at risk" in md
```

---

Better (if you expose value):

```python
assert 0 <= revenue_at_risk <= total_pipeline
```

---

👉 This protects your **CFO-level metric**

---

# 3. ⚠️ Add “No Active Deals” Scenario (Important)

You already fallback in code.

Test it.

---

### Example:

```python
def test_report_handles_no_active_deals_gracefully():
    md = build_sev3_report_md(
        temporal_metrics={"deal_metrics_by_deal_id": {}},
        ...
    )

    assert "Top Revenue Risks" in md
```

---

👉 This ensures:

> No crash + still useful output

---

# 4. 🧭 Add “Trigger Breakdown Consistency” Test

You added this feature — test it.

---

```python
def test_trigger_breakdown_matches_total():
    total = 14
    breakdown = [7, 3, 3, 1]

    assert sum(breakdown) == total
```

---

👉 Prevents subtle reporting inconsistencies

---

# 5. 🔍 Add “Priority Ordering” Test (Very High Value)

You sort deals by:

```python
(len(triggers), risk_score)
```

Test that logic explicitly.

---

```python
def test_high_priority_deals_rank_first():
    deals = [...]
    sorted_deals = sorted(...)

    assert sorted_deals[0]["deal_value_usd_current"] >= sorted_deals[-1]["deal_value_usd_current"]
```

---

👉 This ensures:

> Your **decision ranking logic stays correct**

---

# 🧠 **One Strategic Insight (Very Important)**

You are doing something most people don’t:

---

### You are testing:

> “Is the system trustworthy?”

Not just:

> “Does the code run?”

---

That’s exactly aligned with your:

> **Executive Trust Model**

---

# 🚀 **Final Evaluation of Test Suite**

### You are at:

| Category          | Level       |
| ----------------- | ----------- |
| Coverage          | ✅ Strong    |
| Edge cases        | ✅ Strong    |
| Business logic    | ✅ Excellent |
| Output validation | ✅ Excellent |
| Determinism       | ✅ Excellent |

---

### What you are missing (minor):

* full “end-to-end narrative validation”
* a couple integrity checks

---

# 🏁 **Final Verdict**

This is:

> ✅ Production-quality testing mindset
> ✅ Portfolio differentiator
> ✅ Interview gold



In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path

import pytest

from agents.SEv3.orchestrator.utilities.data_loading import _max_interaction_datetime
from agents.SEv3.orchestrator.utilities.enablement import build_rep_nudges_deterministic
from agents.SEv3.orchestrator.utilities.reporting import (
    _coverage_severity,
    _deal_risk_story,
    _fmt_forecast_stability_run_context,
    _fmt_reference_dt_human,
    _fmt_stability_pct,
    _priority_label,
    _priority_score,
    _trigger_meaning,
    build_sev3_report_md,
)
from agents.SEv3.orchestrator.utilities.temporal_metrics import (
    _consecutive_true_ending_at_end,
    _trigger_evolution,
    compute_temporal_metrics,
)

REPO_ROOT = Path(__file__).resolve().parent
DATA_DIR = REPO_ROOT / "agents" / "data"


# --- data_loading ---


def test_max_interaction_datetime_empty():
    latest, invalid = _max_interaction_datetime([])
    assert latest is None
    assert invalid == 0


def test_max_interaction_datetime_valid_z_and_invalid_rows():
    rows = [
        {"datetime": "2025-06-01T12:00:00Z"},
        {"datetime": "2025-07-15T08:00:00Z"},
        {"datetime": None},
        {"datetime": "not-a-date"},
    ]
    latest, invalid = _max_interaction_datetime(rows)
    assert latest is not None
    assert latest.year == 2025 and latest.month == 7
    assert invalid == 2


# --- temporal_metrics (deterministic helpers) ---


def test_consecutive_true_ending_at_end():
    assert _consecutive_true_ending_at_end([True, False, True, True]) == 2
    assert _consecutive_true_ending_at_end([]) == 0


def test_trigger_evolution_new_and_duration():
    empty = _trigger_evolution([])
    assert empty["new_trigger_flag"] is False

    newly_on = _trigger_evolution([False, True])
    assert newly_on["new_trigger_flag"] is True
    assert newly_on["trigger_duration_periods"] == 1

    sustained = _trigger_evolution([True, True, True])
    assert sustained["new_trigger_flag"] is False
    assert sustained["trigger_duration_periods"] == 3


def test_compute_temporal_metrics_empty_inputs_stable_shape():
    out = compute_temporal_metrics(
        deals=[],
        deal_history=[],
        interactions=[],
        signals_lookup={},
        stage_actions=[],
        thresholds={},
        pipeline_snapshots=[],
        sales_reps=[],
        rep_performance_history=[],
    )
    assert set(out.keys()) >= {
        "deal_metrics_by_deal_id",
        "rep_metrics_by_rep_id",
        "pipeline_metrics",
        "pipeline_triggers",
        "forecast_stability",
    }
    assert out["deal_metrics_by_deal_id"] == {}
    assert isinstance(out["forecast_stability"], dict)


@pytest.mark.skipif(not DATA_DIR.exists(), reason="requires agents/data fixtures")
def test_compute_temporal_metrics_fixture_data_smoke():
    from agents.SEv3.orchestrator.utilities.data_loading import load_sev3_inputs

    bundle = load_sev3_inputs(project_root=str(REPO_ROOT), data_dir="agents/data")
    out = compute_temporal_metrics(
        deals=bundle.get("deals") or [],
        deal_history=bundle.get("deal_history") or [],
        interactions=bundle.get("interactions") or [],
        signals_lookup=bundle.get("signals_lookup") or {},
        stage_actions=bundle.get("stage_actions") or [],
        thresholds=bundle.get("thresholds") or {},
        pipeline_snapshots=bundle.get("pipeline_snapshots") or [],
        sales_reps=bundle.get("sales_reps") or [],
        rep_performance_history=bundle.get("rep_performance_history") or [],
    )
    assert out["pipeline_metrics"].get("current_period")
    assert "avg_deal_forecast_stability_std_dev" in (out.get("forecast_stability") or {})


# --- enablement ---


def test_build_rep_nudges_deterministic_follow_up_overdue():
    ref = datetime(2025, 11, 27, 12, 0, 0, tzinfo=timezone.utc)
    interactions = [
        {
            "rep_id": "SR-01",
            "lead_id": "L-100",
            "deal_id": "D-100",
            "datetime": "2025-11-20T10:00:00Z",
            "next_step_completed": False,
            "next_step_promised": "send pricing",
        }
    ]
    nudges = build_rep_nudges_deterministic(
        stalled_deals=[],
        at_risk_deals=[],
        top_priority_leads=[],
        deal_insights=[],
        interactions=interactions,
        deals_lookup={},
        reference_datetime_utc=ref,
        follow_up_overdue_days=3,
    )
    assert any(n.get("nudge_type") == "follow_up_due" for n in nudges)
    assert nudges[0]["rep_id"] == "SR-01"


# --- reporting formatters ---


def test_fmt_stability_percent_is_probability_points_not_x1000_bug():
    assert _fmt_stability_pct(0.089) == "8.9%"
    assert _fmt_stability_pct(0.89) == "89.0%"


def test_fmt_forecast_stability_volatility_tiers():
    assert "low volatility" in _fmt_forecast_stability_run_context(0.05)
    assert "moderate volatility" in _fmt_forecast_stability_run_context(0.15)
    assert "high volatility" in _fmt_forecast_stability_run_context(0.30)


def test_fmt_reference_dt_human():
    assert "UTC" in _fmt_reference_dt_human("2025-11-27T09:30:00+00:00")
    assert _fmt_reference_dt_human("") == "n/a"


def test_coverage_severity_thresholds():
    assert _coverage_severity(0.5, 0.8) == "critical"  # < 0.75 * threshold
    assert _coverage_severity(0.75, 0.8) == "elevated"
    assert _coverage_severity(0.85, 0.8) == "healthy"


def test_priority_score_and_label_monotonicity():
    base = {
        "deal_risk_score_current": 0.5,
        "deal_value_usd_current": 150000,
        "fired_triggers_current": ["a"],
        "probability_trend_current": 0.01,
    }
    high_value = {**base, "deal_value_usd_current": 400000}
    assert _priority_score(high_value) >= _priority_score(base)
    assert _priority_label(0.71) == "HIGH PRIORITY"
    assert _priority_label(0.50) == "MEDIUM PRIORITY"
    assert _priority_label(0.20) == "LOW PRIORITY"


def test_deal_risk_story_branching():
    assert "no recent engagement" in _deal_risk_story(
        {
            "fired_triggers_current": ["deal_high_value_deteriorating", "no_interaction_in_x_days"],
        }
    )
    assert "stalled progression" in _deal_risk_story(
        {"fired_triggers_current": ["deal_high_value_deteriorating", "deal_stalled_beyond_threshold"]}
    )
    assert "multiple risk signals" in _deal_risk_story({"fired_triggers_current": []})


def test_trigger_meaning_unknown_key():
    assert _trigger_meaning("unknown_trigger_xyz") == "signal requires review"


def test_build_sev3_report_md_minimal_temporal():
    md = build_sev3_report_md(
        data_counts={"deals_count": 0, "reps_count": 0, "interactions_count": 0, "deal_history_deals_count": 0},
        temporal_metrics={
            "deal_metrics_by_deal_id": {},
            "rep_metrics_by_rep_id": {},
            "pipeline_metrics": {"current_period": "2025-01", "coverage_ratio_threshold": 0.8},
            "pipeline_triggers": {},
            "forecast_stability": {"avg_deal_forecast_stability_std_dev": 0.0},
        },
        enablement_report_md="",
        validation_warnings=[],
        reference_datetime_utc="2025-01-15T00:00:00+00:00",
    )
    assert "# Sales Enablement Orchestrator v3 (SEv3)" in md
    assert "## Executive Summary" in md
    assert "0.0%" in md or "n/a" in md  # stability display
